# Amazon ML Challenge: cross-encoder reranker (Kaggle, GPU T4 ×2)

The v5/v6 XGBoost pipeline runs locally. This notebook fine-tunes a **multilingual transformer cross-encoder**
on the raw `name | address` pairs and scores the candidates whose decision is still open. The local stacker
(`blend_ce.py`) then combines the cross-encoder score with the XGBoost probability.

**Setup**
1. Settings → Accelerator → **GPU T4 x2**. Settings → **Internet on** (the base model downloads from Hugging Face).
2. Add the dataset with `ce_train.parquet`, `ce_val.parquet` and `ce_test.parquet` (local folder `kaggle/ce_handoff`).
3. Prefer **Save Version → Save & Run All (Commit)**: it runs in the background and keeps the outputs
   even if the browser disconnects. If a GPU fails (e.g. "uncorrectable ECC error"), stop the session
   and start a new one (new machine). A model saved earlier in the same session is reused.
4. Run all. Both GPUs are used through `torchrun --nproc_per_node=2` (DDP for training, a sharded
   inference with one process per GPU). Memory: the texts stay in Arrow, and only one batch at a time
   becomes Python strings, so RAM stays at a few GB.
5. Download `ce_val_scores.parquet`, `ce_test_scores.parquet` (and any `ce_extra*_scores.parquet`) from the output tab (`ce_model/` is saved
   too; attach this notebook's output to a later session to re-score new pairs without retraining).

Base model: `intfloat/multilingual-e5-small` (MIT, 118M parameters). Alternatives via `CE_MODEL`:
`intfloat/multilingual-e5-base` (MIT, 278M, about 3× slower) or `microsoft/mdeberta-v3-base` (MIT).

In [ ]:
import glob
import os
import subprocess
import time

CFG = {
    "CE_MODEL": "intfloat/multilingual-e5-small",
    "CE_MAXLEN": "96",        # tokens for the pair (name | address, both sides)
    "CE_BS": "128",           # training batch per GPU
    "CE_INF_BS": "512",       # inference batch per GPU
    "CE_EPOCHS": "1",
    "CE_WORKERS": "1",        # tokeniser worker per GPU process (Kaggle has 4 CPUs)
    "CE_LR": "6e-5",
    "CE_MAX_TRAIN": "0",      # 0 = all training pairs; set e.g. 600000 for a quick run
}
hits = glob.glob("/kaggle/input/**/ce_train.parquet", recursive=True)
assert hits, "attach the dataset with ce_train/ce_val/ce_test.parquet"
CFG["CE_DATA"] = os.path.dirname(hits[0])
CFG["CE_OUT"] = "/kaggle/working"
# a previous run's model (attach that notebook's output) -> skip training
prev = glob.glob("/kaggle/input/**/ce_model/config.json", recursive=True)
CFG["CE_MODEL_DIR"] = os.path.dirname(prev[0]) if prev else "/kaggle/working/ce_model"
os.environ.update(CFG)
os.environ.update({"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1", "TOKENIZERS_PARALLELISM": "false"})
print(CFG)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
NGPU = torch.cuda.device_count()
print("GPUs visible:", NGPU)

In [ ]:
%%writefile /kaggle/working/ce.py
"""Cross-encoder: fine-tune (DDP) and score pairs (one shard per GPU).
Tokenisation runs in DataLoader workers (overlaps with the GPU); training batches are length-bucketed.
usage: torchrun --nproc_per_node=N ce.py train | infer <name>"""
import os
import sys
import time

import numpy as np
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

E = os.environ
MODEL, DATA, OUT = E["CE_MODEL"], E["CE_DATA"], E["CE_OUT"]
MODEL_DIR = E.get("CE_MODEL_DIR", os.path.join(OUT, "ce_model"))
MAXLEN, BS, INF_BS = int(E.get("CE_MAXLEN", 96)), int(E.get("CE_BS", 128)), int(E.get("CE_INF_BS", 512))
EPOCHS, LR, MAX_TRAIN = float(E.get("CE_EPOCHS", 1)), float(E.get("CE_LR", 6e-5)), int(E.get("CE_MAX_TRAIN", 0))
WORKERS, SAVE_EVERY = int(E.get("CE_WORKERS", 1)), int(E.get("CE_SAVE_EVERY", 1500))


def setup():
    world = int(E.get("WORLD_SIZE", 1))
    if world > 1:
        dist.init_process_group("nccl" if torch.cuda.is_available() and os.name != "nt" else "gloo")
    rank, local = int(E.get("RANK", 0)), int(E.get("LOCAL_RANK", 0))
    dev = torch.device(f"cuda:{local}" if torch.cuda.is_available() else "cpu")
    if dev.type == "cuda":
        torch.cuda.set_device(dev)
    return rank, world, dev


def log(rank, *a):
    if rank == 0:
        print(time.strftime("%H:%M:%S"), *a, flush=True)


class Batches(Dataset):
    """Item i = one pre-planned batch of row indices -> tokenised tensors (runs in a worker process)."""
    def __init__(self, tbl, batches, tok, y=None):
        self.tbl, self.batches, self.tok, self.y = tbl, batches, tok, y

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, i):
        idx = np.sort(self.batches[i])
        sub = self.tbl.take(idx)
        x = dict(self.tok(sub.column("r_text").to_pylist(), sub.column("s_text").to_pylist(),
                          truncation="longest_first", max_length=MAXLEN, padding=True, return_tensors="pt"))
        x["_idx"] = torch.from_numpy(idx)
        if self.y is not None:
            x["_y"] = torch.from_numpy(self.y[idx])
        return x


def loader(ds):
    return DataLoader(ds, batch_size=None, shuffle=False, num_workers=WORKERS, pin_memory=True,
                      prefetch_factor=8 if WORKERS else None)


def lengths(tbl):
    return (pc.utf8_length(tbl.column("r_text")).to_numpy(zero_copy_only=False)
            + pc.utf8_length(tbl.column("s_text")).to_numpy(zero_copy_only=False))


def save(model, tok, world, note):
    m = model.module if world > 1 else model
    m.save_pretrained(MODEL_DIR)
    tok.save_pretrained(MODEL_DIR)
    open(os.path.join(MODEL_DIR, "STATUS"), "w").write(note)


def train(rank, world, dev):
    tbl = pq.read_table(os.path.join(DATA, "ce_train.parquet"), columns=["r_text", "s_text", "label"])
    if MAX_TRAIN:
        tbl = tbl.slice(0, MAX_TRAIN)
    n = tbl.num_rows
    y = tbl.column("label").to_numpy().astype(np.float32)
    ln = lengths(tbl)
    log(rank, f"train pairs {n:,}  positives {int(y.sum()):,}  world={world}")
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).to(dev)
    if world > 1:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[dev.index] if dev.type == "cuda" else None)
    per_ep = n // (BS * world)
    total = int(per_ep * EPOCHS)
    # batch plan: shuffle, bucket by length inside blocks of 64 batches (less padding), shuffle the batch order;
    # the plan is identical on every rank and rank r takes every world-th batch
    plan = []
    for ep in range(int(np.ceil(EPOCHS))):
        rng = np.random.default_rng(ep)
        perm = rng.permutation(n)[: per_ep * BS * world]
        blk, bs = BS * 64, []
        for i in range(0, len(perm), blk):
            b = perm[i:i + blk]
            b = b[np.argsort(ln[b], kind="stable")]
            bs += [b[j:j + BS] for j in range(0, len(b), BS)]
        plan += [bs[k] for k in rng.permutation(len(bs))]
    plan = plan[: total * world][rank::world]
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    lossf = torch.nn.BCEWithLogitsLoss()
    model.train()
    t0, run = time.time(), 0.0
    for step, x in enumerate(loader(Batches(tbl, plan, tok, y)), 1):
        x.pop("_idx")
        yt = x.pop("_y").to(dev, non_blocking=True)
        x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}
        with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            logit = model(**x).logits.squeeze(-1)
        loss = lossf(logit.float(), yt)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        if step % 50 == 0 or step == 1:        # .item() syncs the GPU, so only occasionally
            run = loss.item() if step == 1 else 0.8 * run + 0.2 * loss.item()
        if step % 200 == 0 or step == total:
            el = time.time() - t0
            log(rank, f"step {step}/{total} loss {run:.4f}  {step * BS * world / el:,.0f} pairs/s  "
                      f"eta {(total - step) * el / step / 60:.1f} min")
        if rank == 0 and step % SAVE_EVERY == 0 and step < total:
            save(model, tok, world, f"partial {step}/{total}")
    if rank == 0:
        save(model, tok, world, "complete")
        log(rank, f"saved {MODEL_DIR}  ({(time.time() - t0) / 60:.1f} min)")


@torch.no_grad()
def infer(rank, world, dev, name):
    n = pq.ParquetFile(os.path.join(DATA, f"{name}.parquet")).metadata.num_rows
    lo, hi = n * rank // world, n * (rank + 1) // world
    tbl = pq.read_table(os.path.join(DATA, f"{name}.parquet"), columns=["r_text", "s_text"]).slice(lo, hi - lo)
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(dev).eval()
    if dev.type == "cuda":
        model = model.half()
    order = np.argsort(lengths(tbl), kind="stable")          # length-sorted batches: little padding
    plan = [order[i:i + INF_BS] for i in range(0, len(order), INF_BS)]
    out = np.zeros(len(order), dtype=np.float32)
    t0 = time.time()
    for b, x in enumerate(loader(Batches(tbl, plan, tok))):
        idx = x.pop("_idx").numpy()
        x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}
        out[idx] = model(**x).logits.squeeze(-1).float().cpu().numpy()
        if b and b % 1000 == 0:
            el = time.time() - t0
            log(rank, f"{name}: {b * INF_BS:,}/{len(order):,} per GPU  {world * b * INF_BS / el:,.0f} pairs/s total  "
                      f"eta {(len(plan) - b) * el / b / 60:.1f} min")
    np.save(os.path.join(OUT, f"scores_{name}_{rank:02d}.npy"), out)
    log(rank, f"{name}: done in {(time.time() - t0) / 60:.1f} min")


if __name__ == "__main__":
    rank, world, dev = setup()
    if sys.argv[1] == "train":
        train(rank, world, dev)
    else:
        infer(rank, world, dev, sys.argv[2])
    if world > 1:
        dist.barrier()
        dist.destroy_process_group()

In [ ]:
def run(*args):
    cmd = ["torchrun", f"--nproc_per_node={max(NGPU, 1)}", "/kaggle/working/ce.py", *args]
    print(" ".join(cmd), flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Warning" not in line and "warn(" not in line:
            print(line, end="", flush=True)
    assert p.wait() == 0, f"{args} failed"
    print(f"--- {args} took {(time.time() - t) / 60:.1f} min")


status = os.path.join(os.environ["CE_MODEL_DIR"], "STATUS")
if not os.path.exists(status):
    run("train")
else:   # a model saved earlier in this session (e.g. before a crash) or from an attached earlier run
    print("using existing model", os.environ["CE_MODEL_DIR"], "->", open(status).read())

In [ ]:
# score every pair file in the dataset: ce_val, ce_test and any later ce_extra*.parquet
NAMES = [n for n in ("ce_val", "ce_test") if os.path.exists(os.path.join(CFG["CE_DATA"], n + ".parquet"))]
NAMES += sorted(os.path.basename(f)[:-8] for f in glob.glob(os.path.join(CFG["CE_DATA"], "ce_extra*.parquet")))
for name in NAMES:
    if not os.path.exists(f"/kaggle/working/{name}_scores.parquet"):
        run("infer", name)

In [ ]:
import numpy as np
import pandas as pd


def merge(name, cols):
    df = pd.read_parquet(os.path.join(CFG["CE_DATA"], f"{name}.parquet"), columns=cols)
    parts = sorted(glob.glob(f"/kaggle/working/scores_{name}_*.npy"))
    df["ce_logit"] = np.concatenate([np.load(p) for p in parts])
    assert len(df) == sum(len(np.load(p)) for p in parts)
    df.to_parquet(f"/kaggle/working/{name}_scores.parquet", index=False)
    for p in parts:
        os.remove(p)
    return df


for name in NAMES:
    if glob.glob(f"/kaggle/working/scores_{name}_*.npy"):
        merge(name, ["rid", "s1", "country", "label", "xgb_p"] if name == "ce_val" else ["rid", "s1", "country"])
val = pd.read_parquet("/kaggle/working/ce_val_scores.parquet")
test = pd.read_parquet("/kaggle/working/ce_test_scores.parquet")
print(len(val), len(test), [f for f in os.listdir("/kaggle/working") if f.endswith(".parquet")])

## Quick diagnostics on validation (the real macro-F0.5 stacking runs locally in `blend_ce.py`)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

xl = np.log(np.clip(val.xgb_p, 1e-6, 1 - 1e-6) / (1 - np.clip(val.xgb_p, 1e-6, 1 - 1e-6)))
X = np.c_[xl, val.ce_logit]
oof = np.zeros(len(val))
for tr, te in GroupKFold(5).split(X, val.label, val.rid):
    oof[te] = LogisticRegression(C=1.0).fit(X[tr], val.label.iloc[tr]).predict_proba(X[te])[:, 1]
val["blend"] = oof
print("AUC  xgb %.4f  ce %.4f  blend %.4f" % tuple(roc_auc_score(val.label, s) for s in (val.xgb_p, val.ce_logit, val.blend)))


def top1(score, t):
    """Record-level accuracy of 'best candidate if score >= t else nothing'."""
    d = val.assign(s=score).sort_values("s", ascending=False).groupby("rid").head(1)
    has = val.groupby("rid").label.max().reindex(d.rid).values
    pick = d.s.values >= t
    ok = np.where(pick, d.label.values == 1, has == 0)
    return ok.mean()


for t in (0.5, 0.6, 0.7, 0.8):
    print(f"t={t}: top-1 decision accuracy  xgb {top1(val.xgb_p, t):.4f}  blend {top1(val.blend, t):.4f}")
print(val.groupby("country")[["label"]].mean(), test.groupby("country").size())